In [8]:
%pip install tqdm


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Network Analysis of S&P 500 Returns

This notebook analyzes the correlation network of S&P 500 stocks during different market regimes. 
We compare network topology (density, average correlation) between periods of market stress and calm to understand how systemic risk propagates. Finally, we generate a rolling time-series of these network metrics to use as features in downstream modeling.

In [3]:
import sys
import os

# Add the parent directory (repo root) to the system path
# Find the repo root whether the notebook is opened from the repo root or from notebooks/
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
sys.path.append(str(ROOT))

# Now you can import from src normally
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from src.windows import correlation_at

def analyze_network(returns, target_date, threshold=0.55):
    """
    Builds and analyzes a correlation network for a specific 60-day window.
    """
    # Generate the correlation matrix
    corr_matrix = correlation_at(returns, end=target_date, window=60)
    
    # ISSUE 1 FIX: Calculate average correlation correctly without weighting issues
    corr_array = corr_matrix.to_numpy()
    avg_correlation = corr_array[np.triu_indices(len(corr_matrix), k=1)].mean()
    
    # ISSUE 3 FIX: Use positive correlations only for "stocks moving together"
    adjacency_matrix = (corr_matrix > threshold).astype(int)
    
    # Create network
    G = nx.from_pandas_adjacency(adjacency_matrix)
    G.remove_edges_from(nx.selfloop_edges(G))
    
    # ISSUE 2 FIX: Calculate density on ALL assets before removing isolates
    network_density = nx.density(G)
    
    # Remove isolated nodes for a cleaner visualization
    isolates = list(nx.isolates(G))
    G.remove_nodes_from(isolates)
    
    print(f"--- Target Date: {target_date} ---")
    print(f"Network Density (all assets): {network_density:.4f}")
    print(f"Average 60-day Correlation: {avg_correlation:.4f}")
    print(f"Nodes with edges (Threshold > {threshold}): {G.number_of_nodes()}")
    print(f"Total Edges: {G.number_of_edges()}\n")
    
    return G

# Load the processed daily returns
try:
    returns = pd.read_parquet(ROOT / 'data' / 'processed' / 'returns_daily.parquet')
except FileNotFoundError:
    print("Processed returns not found. Please verify the path.")

# ISSUE 4 FIX: Compare a stress period vs a calm period
print("Comparing Market Regimes:\n")
# Stress Period (e.g., 2008 Financial Crisis)
G_stress = analyze_network(returns, target_date='2008-10-15', threshold=0.55)

# Calm Period (e.g., mid-2017 low volatility environment)
G_calm = analyze_network(returns, target_date='2017-10-15', threshold=0.55)

Comparing Market Regimes:

--- Target Date: 2008-10-15 ---
Network Density (all assets): 0.6287
Average 60-day Correlation: 0.5804
Nodes with edges (Threshold > 0.55): 300
Total Edges: 28195

--- Target Date: 2017-10-15 ---
Network Density (all assets): 0.0160
Average 60-day Correlation: 0.1289
Nodes with edges (Threshold > 0.55): 323
Total Edges: 1300



### Analysis of Results
During the 2008 financial crisis, the network shows a dramatic increase in both average correlation and network density compared to the 2017 calm period. This confirms that during market stress, equities become highly interconnected, moving together tightly as systemic risk dominates idiosyncratic stock behavior.

In [10]:
# ISSUE 7 FIX: Generate historical network metrics feature table
from tqdm import tqdm

def generate_historical_network_features(returns, window=60, threshold=0.55, freq='W'):
    """
    Computes rolling network metrics over the full history.
    freq='W' samples weekly to save computation time while maintaining granularity.
    """
    print(f"Generating network features (Window: {window}, Threshold: {threshold})...")
    
    # Sample dates to avoid computing N^2 operations for every single trading day
    # 'W-FRI' means weekly on Fridays
    eval_dates = returns.resample(freq).last().dropna(how='all').index
    
    records = []
    for date in tqdm(eval_dates, desc="Processing Windows"):
        # We need at least 'window' days of data prior to the date
        if len(returns.loc[:date]) < window:
            continue
            
        corr_matrix = correlation_at(returns, end=date, window=window)
        
        # Calculate Average Correlation
        corr_array = corr_matrix.to_numpy()
        avg_corr = corr_array[np.triu_indices(len(corr_matrix), k=1)].mean()
        
        # Calculate Network Density (all assets included)
        adjacency_matrix = (corr_matrix > threshold).astype(int)
        G = nx.from_pandas_adjacency(adjacency_matrix)
        G.remove_edges_from(nx.selfloop_edges(G))
        density = nx.density(G)
        
        records.append({
            'date': date,
            'network_density': density,
            'avg_correlation': avg_corr
        })
        
    features_df = pd.DataFrame(records).set_index('date')
    return features_df

# Generate and save the feature table
network_features = generate_historical_network_features(returns)

# Display head of the newly created features
display(network_features.head())

# Save to processed data folder
output_path = ROOT / 'data' / 'processed' / 'network_features.parquet'
network_features.to_parquet(output_path)
print(f"Saved historical network features to {output_path}")

Generating network features (Window: 60, Threshold: 0.55)...


Processing Windows:   0%|          | 0/1140 [00:00<?, ?it/s]

Processing Windows: 100%|██████████| 1140/1140 [00:24<00:00, 47.37it/s]


,network_density,avg_correlation
date,,
2005-04-03,0.026094,0.246936
2005-04-10,0.026848,0.249777
2005-04-17,0.038574,0.274936
2005-04-24,0.053580,0.299147
2005-05-01,0.063071,0.309647


Saved historical network features to ../data/processed/network_features.parquet
